# 🌍 LG전자 글로벌 경쟁사 실시간 데이터 분석

## 📋 프로젝트 개요
- **목적**: 무료 API를 활용한 LG전자와 글로벌 경쟁사들의 실시간 데이터 수집 및 분석
- **데이터 소스**: Yahoo Finance, Alpha Vantage, SEC API 등
- **분석 범위**: 미국, 유럽, 일본, 중국 주요 가전/전자 기업

## 🎯 분석 대상 기업
### 🇰🇷 **한국**
- **LG전자** (066570.KS)

### 🇺🇸 **미국**
- **Whirlpool** (WHR) - 가전
- **General Electric** (GE) - 전자/가전

### 🇪🇺 **유럽**
- **Electrolux** (ELUX-B.ST) - 스웨덴 가전
- **BSH** (비상장) → **Siemens** (SIE.DE) 대체
- **Miele** (비상장) → **Philips** (PHIA.AS) 대체

### 🇯🇵 **일본**
- **Panasonic** (6752.T)
- **Sharp** (6753.T)

### 🇨🇳 **중국**
- **Haier** (600690.SS)
- **Midea** (000333.SZ)

## 📊 수집할 데이터 항목
1. **주가 정보**: 현재가, 시가총액, 거래량
2. **재무 지표**: P/E, P/B, ROE, 부채비율
3. **성과 지표**: 매출액, 영업이익, 순이익
4. **기술적 지표**: 52주 최고/최저, 베타

In [1]:
# 필요한 라이브러리 설치 및 임포트
import warnings
warnings.filterwarnings('ignore')

# 기본 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# API 관련 라이브러리
import yfinance as yf
import requests
import json
from datetime import datetime, timedelta
import time

# 환경변수 관리
import os
from dotenv import load_dotenv

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("📦 라이브러리 로딩 완료!")
print("🌍 글로벌 경쟁사 실시간 데이터 분석을 시작합니다!")
print("=" * 60)

📦 라이브러리 로딩 완료!
🌍 글로벌 경쟁사 실시간 데이터 분석을 시작합니다!


In [2]:
# .env 파일 로드 (API 키 설정)
load_dotenv()

# API 키 확인 (설정되어 있다면)
alpha_vantage_key = os.getenv('ALPHA_VANTAGE_API_KEY')
sec_user_agent = os.getenv('SEC_USER_AGENT', 'lg-analysis@example.com')

print("🔑 API 설정 확인:")
print(f"   • Alpha Vantage API: {'✅ 설정됨' if alpha_vantage_key else '❌ 미설정 (Yahoo Finance 사용)'}")
print(f"   • SEC User Agent: {sec_user_agent}")
print("\n💡 참고: .env 파일이 없어도 Yahoo Finance로 분석 가능합니다!")

🔑 API 설정 확인:
   • Alpha Vantage API: ❌ 미설정 (Yahoo Finance 사용)
   • SEC User Agent: lg-analysis@example.com

💡 참고: .env 파일이 없어도 Yahoo Finance로 분석 가능합니다!


In [3]:
# 글로벌 경쟁사 데이터 정의
companies_info = {
    '회사명': [
        'LG전자', 'Whirlpool', 'General Electric', 'Electrolux', 
        'Siemens', 'Philips', 'Panasonic', 'Sharp', 'Haier', 'Midea'
    ],
    '티커': [
        '066570.KS', 'WHR', 'GE', 'ELUX-B.ST',
        'SIE.DE', 'PHIA.AS', '6752.T', '6753.T', '600690.SS', '000333.SZ'
    ],
    '국가': [
        '한국', '미국', '미국', '스웨덴',
        '독일', '네덜란드', '일본', '일본', '중국', '중국'
    ],
    '주요사업': [
        '가전/전자', '가전', '전자/에너지', '가전',
        '산업/헬스케어', '헬스케어/가전', '전자/가전', '전자/디스플레이', '가전', '가전/HVAC'
    ]
}

companies_df = pd.DataFrame(companies_info)

print("🏢 분석 대상 글로벌 경쟁사 목록:")
print("-" * 50)
for idx, row in companies_df.iterrows():
    flag = {'한국': '🇰🇷', '미국': '🇺🇸', '스웨덴': '🇸🇪', '독일': '🇩🇪', 
            '네덜란드': '🇳🇱', '일본': '🇯🇵', '중국': '🇨🇳'}.get(row['국가'], '🌍')
    print(f"   {idx+1:2d}. {flag} {row['회사명']} ({row['티커']}) - {row['주요사업']}")

print(f"\n📊 총 {len(companies_df)}개 기업 분석 예정")

🏢 분석 대상 글로벌 경쟁사 목록:
--------------------------------------------------
    1. 🇰🇷 LG전자 (066570.KS) - 가전/전자
    2. 🇺🇸 Whirlpool (WHR) - 가전
    3. 🇺🇸 General Electric (GE) - 전자/에너지
    4. 🇸🇪 Electrolux (ELUX-B.ST) - 가전
    5. 🇩🇪 Siemens (SIE.DE) - 산업/헬스케어
    6. 🇳🇱 Philips (PHIA.AS) - 헬스케어/가전
    7. 🇯🇵 Panasonic (6752.T) - 전자/가전
    8. 🇯🇵 Sharp (6753.T) - 전자/디스플레이
    9. 🇨🇳 Haier (600690.SS) - 가전
   10. 🇨🇳 Midea (000333.SZ) - 가전/HVAC

📊 총 10개 기업 분석 예정


In [4]:
# 실시간 주가 및 기본 정보 수집 함수
def get_stock_data(ticker):
    """Yahoo Finance를 통한 주가 및 기본 정보 수집"""
    try:
        stock = yf.Ticker(ticker)
        
        # 기본 정보
        info = stock.info
        
        # 주가 데이터 (최근 1개월)
        hist = stock.history(period="1mo")
        current_price = hist['Close'].iloc[-1] if not hist.empty else None
        
        # 재무 데이터 수집
        data = {
            'ticker': ticker,
            'current_price': current_price,
            'currency': info.get('currency', 'N/A'),
            'market_cap': info.get('marketCap', None),
            'pe_ratio': info.get('trailingPE', None),
            'pb_ratio': info.get('priceToBook', None),
            'roe': info.get('returnOnEquity', None),
            'debt_to_equity': info.get('debtToEquity', None),
            'revenue': info.get('totalRevenue', None),
            'gross_profit': info.get('grossProfits', None),
            'operating_margin': info.get('operatingMargins', None),
            'profit_margin': info.get('profitMargins', None),
            'beta': info.get('beta', None),
            '52week_high': info.get('fiftyTwoWeekHigh', None),
            '52week_low': info.get('fiftyTwoWeekLow', None),
            'volume': info.get('volume', None),
            'avg_volume': info.get('averageVolume', None),
            'dividend_yield': info.get('dividendYield', None),
            'sector': info.get('sector', 'N/A'),
            'industry': info.get('industry', 'N/A'),
            'employees': info.get('fullTimeEmployees', None),
            'country': info.get('country', 'N/A')
        }
        
        return data
        
    except Exception as e:
        print(f"❌ {ticker} 데이터 수집 실패: {str(e)}")
        return None

# 데이터 수집 실행
print("📡 실시간 데이터 수집 중...")
print("-" * 40)

all_stock_data = []
success_count = 0
total_count = len(companies_df)

for idx, row in companies_df.iterrows():
    print(f"   {idx+1:2d}/{total_count} {row['회사명']} ({row['티커']}) 수집 중...", end=" ")
    
    data = get_stock_data(row['티커'])
    if data:
        data['회사명'] = row['회사명']
        data['국가'] = row['국가']
        data['주요사업'] = row['주요사업']
        all_stock_data.append(data)
        success_count += 1
        print("✅")
    else:
        print("❌")
    
    # API 호출 제한 방지
    time.sleep(0.5)

print(f"\n📊 데이터 수집 완료: {success_count}/{total_count}개 성공")

# DataFrame 생성
if all_stock_data:
    stocks_df = pd.DataFrame(all_stock_data)
    print(f"📋 수집된 데이터 컬럼 수: {len(stocks_df.columns)}개")
else:
    print("❌ 수집된 데이터가 없습니다.")

📡 실시간 데이터 수집 중...
----------------------------------------
    1/10 LG전자 (066570.KS) 수집 중... ✅
✅
    2/10 Whirlpool (WHR) 수집 중...     2/10 Whirlpool (WHR) 수집 중... ✅
✅
    3/10 General Electric (GE) 수집 중...     3/10 General Electric (GE) 수집 중... ✅
✅
    4/10 Electrolux (ELUX-B.ST) 수집 중...     4/10 Electrolux (ELUX-B.ST) 수집 중... ✅
✅
    5/10 Siemens (SIE.DE) 수집 중...     5/10 Siemens (SIE.DE) 수집 중... ✅
✅
    6/10 Philips (PHIA.AS) 수집 중...     6/10 Philips (PHIA.AS) 수집 중... ✅
✅
    7/10 Panasonic (6752.T) 수집 중...     7/10 Panasonic (6752.T) 수집 중... ✅
✅
    8/10 Sharp (6753.T) 수집 중...     8/10 Sharp (6753.T) 수집 중... ✅
✅
    9/10 Haier (600690.SS) 수집 중...     9/10 Haier (600690.SS) 수집 중... ✅
✅
   10/10 Midea (000333.SZ) 수집 중...    10/10 Midea (000333.SZ) 수집 중... ✅
✅

📊 데이터 수집 완료: 10/10개 성공
📋 수집된 데이터 컬럼 수: 25개

📊 데이터 수집 완료: 10/10개 성공
📋 수집된 데이터 컬럼 수: 25개


In [5]:
# 수집된 데이터 정리 및 전처리
if 'stocks_df' in locals() and not stocks_df.empty:
    # 데이터 정리
    stocks_clean = stocks_df.copy()
    
    # 시가총액을 USD 기준으로 통일 (대략적 환율 적용)
    exchange_rates = {
        'KRW': 0.00075,  # 원화
        'SEK': 0.092,    # 스웨덴 크로나
        'EUR': 1.08,     # 유로
        'JPY': 0.0067,   # 엔화
        'CNY': 0.14,     # 위안화
        'USD': 1.0       # 달러
    }
    
    # 시가총액 USD 변환
    stocks_clean['market_cap_usd'] = stocks_clean.apply(
        lambda row: row['market_cap'] * exchange_rates.get(row['currency'], 1.0) 
        if pd.notna(row['market_cap']) else None, axis=1
    )
    
    # 백분율 데이터 변환 (소수점 → 백분율)
    percentage_cols = ['roe', 'operating_margin', 'profit_margin', 'dividend_yield']
    for col in percentage_cols:
        stocks_clean[col] = stocks_clean[col] * 100  # 소수점을 백분율로 변환
    
    # 데이터 요약 출력
    print("📊 수집된 데이터 요약:")
    print("-" * 40)
    print(f"   • 총 기업 수: {len(stocks_clean)}개")
    print(f"   • 시가총액 데이터: {stocks_clean['market_cap_usd'].notna().sum()}개")
    print(f"   • P/E 비율 데이터: {stocks_clean['pe_ratio'].notna().sum()}개")
    print(f"   • ROE 데이터: {stocks_clean['roe'].notna().sum()}개")
    
    # 상위 5개 기업 미리보기
    print(f"\n📋 상위 데이터 미리보기:")
    display_cols = ['회사명', '국가', 'current_price', 'currency', 'market_cap_usd', 'pe_ratio', 'roe']
    display(stocks_clean[display_cols].head())
    
else:
    print("❌ 분석할 데이터가 없습니다.")

📊 수집된 데이터 요약:
----------------------------------------
   • 총 기업 수: 10개
   • 시가총액 데이터: 10개
   • P/E 비율 데이터: 8개
   • ROE 데이터: 9개

📋 상위 데이터 미리보기:


,회사명,국가,current_price,currency,market_cap_usd,pe_ratio,roe
0,LG전자,한국,75700.000000,KRW,9.735493e+09,NaN,NaN
1,Whirlpool,미국,83.040001,USD,4.641471e+09,NaN,-4.106000
2,General Electric,미국,271.079987,USD,2.874641e+11,38.72571,39.567003
3,Electrolux,스웨덴,59.820000,SEK,1.502681e+09,122.08163,1.476000
4,Siemens,독일,224.899994,EUR,1.902371e+11,22.85569,14.993000


In [6]:
# 1. 시가총액 비교 시각화
if 'stocks_clean' in locals() and not stocks_clean.empty:
    print("📈 1. 글로벌 경쟁사 시가총액 비교")
    print("-" * 40)
    
    # 시가총액 데이터가 있는 기업만 필터링
    market_cap_data = stocks_clean[stocks_clean['market_cap_usd'].notna()].copy()
    market_cap_data = market_cap_data.sort_values('market_cap_usd', ascending=False)
    
    if not market_cap_data.empty:
        # 시가총액을 억 달러 단위로 변환
        market_cap_data['market_cap_billion'] = market_cap_data['market_cap_usd'] / 1e9
        
        # LG전자 하이라이트 색상
        colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                 for company in market_cap_data['회사명']]
        
        # Plotly 바차트 생성
        fig1 = go.Figure(data=[
            go.Bar(
                x=market_cap_data['회사명'],
                y=market_cap_data['market_cap_billion'],
                text=market_cap_data['market_cap_billion'].apply(lambda x: f'${x:.1f}B'),
                textposition='auto',
                marker_color=colors,
                hovertemplate='<b>%{x}</b><br>시가총액: $%{y:.1f}B<br>국가: %{customdata}<extra></extra>',
                customdata=market_cap_data['국가']
            )
        ])
        
        fig1.update_layout(
            title={
                'text': '🌍 글로벌 경쟁사 시가총액 비교 (실시간 데이터)',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 20, 'family': 'Arial Black'}
            },
            xaxis_title='회사명',
            yaxis_title='시가총액 (Billion USD)',
            plot_bgcolor='white',
            height=500,
            showlegend=False
        )
        
        fig1.update_xaxes(tickangle=45)
        fig1.show()
        
        # 순위 출력
        print(f"\n🏆 시가총액 순위 (실시간):")
        for i, (_, row) in enumerate(market_cap_data.iterrows(), 1):
            symbol = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}위"
            highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
            flag = {'한국': '🇰🇷', '미국': '🇺🇸', '스웨덴': '🇸🇪', '독일': '🇩🇪', 
                   '네덜란드': '🇳🇱', '일본': '🇯🇵', '중국': '🇨🇳'}.get(row['국가'], '🌍')
            print(f"   {symbol} {flag} {row['회사명']}: ${row['market_cap_billion']:.1f}B{highlight}")
        
        # LG전자 포지션 분석
        if 'LG전자' in market_cap_data['회사명'].values:
            lg_rank = market_cap_data.reset_index(drop=True)[
                market_cap_data.reset_index(drop=True)['회사명'] == 'LG전자'
            ].index[0] + 1
            lg_market_cap = market_cap_data[market_cap_data['회사명'] == 'LG전자']['market_cap_billion'].iloc[0]
            total_companies = len(market_cap_data)
            
            print(f"\n📊 LG전자 시가총액 포지션:")
            print(f"   • 순위: {lg_rank}/{total_companies}위")
            print(f"   • 시가총액: ${lg_market_cap:.1f}B")
            print(f"   • 상위 비율: {((total_companies - lg_rank + 1) / total_companies * 100):.1f}%")
    
    else:
        print("❌ 시가총액 데이터를 가진 기업이 없습니다.")

📈 1. 글로벌 경쟁사 시가총액 비교
----------------------------------------



🏆 시가총액 순위 (실시간):
   🥇 🇺🇸 General Electric: $287.5B
   🥈 🇩🇪 Siemens: $190.2B
   🥉 🇨🇳 Midea: $77.6B
   4위 🇨🇳 Haier: $30.8B
   5위 🇳🇱 Philips: $24.3B
   6위 🇯🇵 Panasonic: $23.6B
   7위 🇰🇷 LG전자: $9.7B ⭐
   8위 🇺🇸 Whirlpool: $4.6B
   9위 🇯🇵 Sharp: $3.2B
   10위 🇸🇪 Electrolux: $1.5B

📊 LG전자 시가총액 포지션:
   • 순위: 7/10위
   • 시가총액: $9.7B
   • 상위 비율: 40.0%


In [8]:
# 2. 밸류에이션 지표 비교 (P/E, P/B)
if 'stocks_clean' in locals() and not stocks_clean.empty:
    print("\n📈 2. 밸류에이션 지표 비교 (P/E, P/B)")
    print("-" * 40)
    
    # P/E와 P/B 데이터가 있는 기업 필터링
    valuation_data = stocks_clean[
        (stocks_clean['pe_ratio'].notna()) | (stocks_clean['pb_ratio'].notna())
    ].copy()
    
    if not valuation_data.empty:
        # 서브플롯 생성
        fig2 = make_subplots(
            rows=1, cols=2,
            subplot_titles=('P/E Ratio (실시간)', 'P/B Ratio (실시간)'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        # P/E Ratio (이상치 제거: 100 이상은 제외)
        pe_data = valuation_data[
            (valuation_data['pe_ratio'].notna()) & (valuation_data['pe_ratio'] < 100)
        ]
        
        if not pe_data.empty:
            # P/E용 색상
            pe_colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                        for company in pe_data['회사명']]
            
            fig2.add_trace(
                go.Bar(
                    x=pe_data['회사명'], 
                    y=pe_data['pe_ratio'], 
                    name='P/E',
                    marker_color=pe_colors,
                    text=pe_data['pe_ratio'].apply(lambda x: f'{x:.1f}x'),
                    textposition='auto',
                    hovertemplate='<b>%{x}</b><br>P/E: %{y:.1f}x<extra></extra>'
                ),
                row=1, col=1
            )
        
        # P/B Ratio
        pb_data = valuation_data[valuation_data['pb_ratio'].notna()]
        
        if not pb_data.empty:
            # P/B용 색상
            pb_colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                        for company in pb_data['회사명']]
            
            fig2.add_trace(
                go.Bar(
                    x=pb_data['회사명'], 
                    y=pb_data['pb_ratio'], 
                    name='P/B',
                    marker_color=pb_colors,
                    text=pb_data['pb_ratio'].apply(lambda x: f'{x:.1f}x'),
                    textposition='auto',
                    hovertemplate='<b>%{x}</b><br>P/B: %{y:.1f}x<extra></extra>'
                ),
                row=1, col=2
            )
        
        fig2.update_layout(
            title={
                'text': '💰 글로벌 경쟁사 밸류에이션 비교 (실시간)',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 18, 'family': 'Arial Black'}
            },
            height=500,
            showlegend=False
        )
        
        fig2.update_xaxes(tickangle=45, row=1, col=1)
        fig2.update_xaxes(tickangle=45, row=1, col=2)
        fig2.show()
        
        # 밸류에이션 순위 출력
        if not pe_data.empty:
            pe_sorted = pe_data.sort_values('pe_ratio')
            print(f"\n📊 P/E Ratio 순위 (낮을수록 저평가):")
            for i, (_, row) in enumerate(pe_sorted.iterrows(), 1):
                highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
                flag = {'한국': '🇰🇷', '미국': '🇺🇸', '스웨덴': '🇸🇪', '독일': '🇩🇪', 
                       '네덜란드': '🇳🇱', '일본': '🇯🇵', '중국': '🇨🇳'}.get(row['국가'], '🌍')
                print(f"   {i}. {flag} {row['회사명']}: {row['pe_ratio']:.1f}x{highlight}")
        
        if not pb_data.empty:
            pb_sorted = pb_data.sort_values('pb_ratio')
            print(f"\n📊 P/B Ratio 순위 (낮을수록 저평가):")
            for i, (_, row) in enumerate(pb_sorted.iterrows(), 1):
                highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
                flag = {'한국': '🇰🇷', '미국': '🇺🇸', '스웨덴': '🇸🇪', '독일': '🇩🇪', 
                       '네덜란드': '🇳🇱', '일본': '🇯🇵', '중국': '🇨🇳'}.get(row['국가'], '🌍')
                print(f"   {i}. {flag} {row['회사명']}: {row['pb_ratio']:.1f}x{highlight}")
    
    else:
        print("❌ 밸류에이션 데이터를 가진 기업이 없습니다.")


📈 2. 밸류에이션 지표 비교 (P/E, P/B)
----------------------------------------



📊 P/E Ratio 순위 (낮을수록 저평가):
   1. 🇯🇵 Panasonic: 9.3x
   2. 🇨🇳 Haier: 11.9x
   3. 🇨🇳 Midea: 12.2x
   4. 🇯🇵 Sharp: 13.2x
   5. 🇩🇪 Siemens: 22.9x
   6. 🇺🇸 General Electric: 38.7x

📊 P/B Ratio 순위 (낮을수록 저평가):
   1. 🇯🇵 Panasonic: 0.7x
   2. 🇨🇳 Haier: 2.0x
   3. 🇺🇸 Whirlpool: 2.0x
   4. 🇸🇪 Electrolux: 2.0x
   5. 🇳🇱 Philips: 2.1x
   6. 🇨🇳 Midea: 2.3x
   7. 🇩🇪 Siemens: 3.1x
   8. 🇯🇵 Sharp: 3.1x
   9. 🇺🇸 General Electric: 15.0x


In [9]:
# 3. 수익성 지표 분석 (ROE, 영업이익률, 순이익률)
if 'stocks_clean' in locals() and not stocks_clean.empty:
    print("\n📈 3. 수익성 지표 분석")
    print("-" * 40)
    
    # 수익성 데이터가 있는 기업 필터링
    profitability_data = stocks_clean[
        (stocks_clean['roe'].notna()) | 
        (stocks_clean['operating_margin'].notna()) | 
        (stocks_clean['profit_margin'].notna())
    ].copy()
    
    if not profitability_data.empty:
        # 서브플롯 생성
        fig3 = make_subplots(
            rows=2, cols=2,
            subplot_titles=('ROE (%)', '영업이익률 (%)', '순이익률 (%)', '수익성 종합 비교'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"type": "xy"}]]
        )
        
        colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                 for company in profitability_data['회사명']]
        
        # ROE
        roe_data = profitability_data[profitability_data['roe'].notna()]
        if not roe_data.empty:
            roe_colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                         for company in roe_data['회사명']]
            
            fig3.add_trace(
                go.Bar(
                    x=roe_data['회사명'], 
                    y=roe_data['roe'], 
                    name='ROE',
                    marker_color=roe_colors,
                    text=roe_data['roe'].apply(lambda x: f'{x:.1f}%'),
                    textposition='auto'
                ),
                row=1, col=1
            )
        
        # 영업이익률
        operating_data = profitability_data[profitability_data['operating_margin'].notna()]
        if not operating_data.empty:
            op_colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                        for company in operating_data['회사명']]
            
            fig3.add_trace(
                go.Bar(
                    x=operating_data['회사명'], 
                    y=operating_data['operating_margin'], 
                    name='영업이익률',
                    marker_color=op_colors,
                    text=operating_data['operating_margin'].apply(lambda x: f'{x:.1f}%'),
                    textposition='auto'
                ),
                row=1, col=2
            )
        
        # 순이익률
        profit_data = profitability_data[profitability_data['profit_margin'].notna()]
        if not profit_data.empty:
            profit_colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                           for company in profit_data['회사명']]
            
            fig3.add_trace(
                go.Bar(
                    x=profit_data['회사명'], 
                    y=profit_data['profit_margin'], 
                    name='순이익률',
                    marker_color=profit_colors,
                    text=profit_data['profit_margin'].apply(lambda x: f'{x:.1f}%'),
                    textposition='auto'
                ),
                row=2, col=1
            )
        
        # 수익성 종합 비교 (ROE vs 영업이익률)
        comprehensive_data = profitability_data[
            (profitability_data['roe'].notna()) & 
            (profitability_data['operating_margin'].notna())
        ]
        
        if not comprehensive_data.empty:
            comp_colors = ['#e74c3c' if company == 'LG전자' else '#3498db' 
                          for company in comprehensive_data['회사명']]
            
            fig3.add_trace(
                go.Scatter(
                    x=comprehensive_data['operating_margin'],
                    y=comprehensive_data['roe'],
                    mode='markers+text',
                    text=comprehensive_data['회사명'],
                    textposition="top center",
                    marker=dict(
                        size=12,
                        color=comp_colors,
                        opacity=0.7,
                        line=dict(width=2, color='DarkSlateGrey')
                    ),
                    name='종합 수익성',
                    showlegend=False
                ),
                row=2, col=2
            )
        
        fig3.update_layout(
            title={
                'text': '💎 글로벌 경쟁사 수익성 분석 (실시간)',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 18, 'family': 'Arial Black'}
            },
            height=800,
            showlegend=False
        )
        
        # 각 서브플롯 축 업데이트
        fig3.update_xaxes(tickangle=45, row=1, col=1)
        fig3.update_xaxes(tickangle=45, row=1, col=2)
        fig3.update_xaxes(tickangle=45, row=2, col=1)
        fig3.update_xaxes(title_text="영업이익률 (%)", row=2, col=2)
        fig3.update_yaxes(title_text="ROE (%)", row=2, col=2)
        
        fig3.show()
        
        # 수익성 순위 출력
        if not roe_data.empty:
            roe_sorted = roe_data.sort_values('roe', ascending=False)
            print(f"\n🏆 ROE 순위:")
            for i, (_, row) in enumerate(roe_sorted.iterrows(), 1):
                symbol = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}위"
                highlight = " ⭐" if row['회사명'] == 'LG전자' else ""
                flag = {'한국': '🇰🇷', '미국': '🇺🇸', '스웨덴': '🇸🇪', '독일': '🇩🇪', 
                       '네덜란드': '🇳🇱', '일본': '🇯🇵', '중국': '🇨🇳'}.get(row['국가'], '🌍')
                print(f"   {symbol} {flag} {row['회사명']}: {row['roe']:.1f}%{highlight}")
    
    else:
        print("❌ 수익성 데이터를 가진 기업이 없습니다.")


📈 3. 수익성 지표 분석
----------------------------------------



🏆 ROE 순위:
   🥇 🇺🇸 General Electric: 39.6%
   🥈 🇯🇵 Sharp: 22.0%
   🥉 🇨🇳 Midea: 20.0%
   4위 🇨🇳 Haier: 17.2%
   5위 🇩🇪 Siemens: 15.0%
   6위 🇯🇵 Panasonic: 7.8%
   7위 🇳🇱 Philips: 1.5%
   8위 🇸🇪 Electrolux: 1.5%
   9위 🇺🇸 Whirlpool: -4.1%


In [10]:
# 4. 국가별 포트폴리오 분석
if 'stocks_clean' in locals() and not stocks_clean.empty:
    print("\n📈 4. 국가별 포트폴리오 분석")
    print("-" * 40)
    
    # 국가별 집계
    country_analysis = stocks_clean.groupby('국가').agg({
        '회사명': 'count',
        'market_cap_usd': ['mean', 'sum'],
        'pe_ratio': 'mean',
        'pb_ratio': 'mean',
        'roe': 'mean',
        'operating_margin': 'mean'
    }).round(2)
    
    # 컬럼명 정리
    country_analysis.columns = [
        '기업수', '평균시가총액', '총시가총액', '평균PE', '평균PB', '평균ROE', '평균영업이익률'
    ]
    
    # 국가별 시가총액 비교 (파이 차트)
    country_market_cap = stocks_clean.groupby('국가')['market_cap_usd'].sum().dropna()
    
    if not country_market_cap.empty:
        fig4 = make_subplots(
            rows=1, cols=2,
            subplot_titles=('국가별 시가총액 분포', '국가별 평균 ROE'),
            specs=[[{"type": "domain"}, {"secondary_y": False}]]
        )
        
        # 파이 차트
        fig4.add_trace(
            go.Pie(
                labels=country_market_cap.index,
                values=country_market_cap.values,
                name="시가총액",
                textinfo='label+percent',
                hovertemplate='<b>%{label}</b><br>시가총액: $%{value:.1e}<br>비율: %{percent}<extra></extra>'
            ),
            row=1, col=1
        )
        
        # 국가별 평균 ROE
        country_roe = stocks_clean.groupby('국가')['roe'].mean().dropna()
        if not country_roe.empty:
            colors_country = ['#e74c3c' if country == '한국' else '#3498db' 
                             for country in country_roe.index]
            
            fig4.add_trace(
                go.Bar(
                    x=country_roe.index,
                    y=country_roe.values,
                    name='평균 ROE',
                    marker_color=colors_country,
                    text=country_roe.values.round(1),
                    textposition='auto'
                ),
                row=1, col=2
            )
        
        fig4.update_layout(
            title={
                'text': '🌍 국가별 경쟁력 분석',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': 18, 'family': 'Arial Black'}
            },
            height=500
        )
        
        fig4.update_yaxes(title_text="ROE (%)", row=1, col=2)
        fig4.show()
    
    # 국가별 상세 분석 테이블
    print(f"\n📊 국가별 상세 분석:")
    display(country_analysis)
    
    # 한국(LG전자) 포지션 분석
    if '한국' in country_analysis.index:
        korea_data = country_analysis.loc['한국']
        print(f"\n🇰🇷 한국(LG전자) 글로벌 포지션:")
        print(f"   • 평균 시가총액: ${korea_data['평균시가총액']:.1e}")
        print(f"   • 평균 P/E: {korea_data['평균PE']:.1f}x")
        print(f"   • 평균 P/B: {korea_data['평균PB']:.1f}x")
        print(f"   • 평균 ROE: {korea_data['평균ROE']:.1f}%")
        print(f"   • 평균 영업이익률: {korea_data['평균영업이익률']:.1f}%")


📈 4. 국가별 포트폴리오 분석
----------------------------------------



📊 국가별 상세 분석:


,기업수,평균시가총액,총시가총액,평균PE,평균PB,평균ROE,평균영업이익률
국가,,,,,,,
네덜란드,1,2.433982e+10,2.433982e+10,144.38,2.12,1.50,9.24
독일,1,1.902371e+11,1.902371e+11,22.86,3.08,14.99,11.78
미국,2,1.460528e+11,2.921055e+11,38.73,8.50,17.73,13.11
스웨덴,1,1.502681e+09,1.502681e+09,122.08,2.00,1.48,2.55
일본,2,1.341454e+10,2.682908e+10,11.28,1.92,14.92,3.01
중국,2,5.417278e+10,1.083456e+11,12.06,2.14,18.60,9.03
한국,1,9.735493e+09,9.735493e+09,NaN,NaN,NaN,3.08



🇰🇷 한국(LG전자) 글로벌 포지션:
   • 평균 시가총액: $9.7e+09
   • 평균 P/E: nanx
   • 평균 P/B: nanx
   • 평균 ROE: nan%
   • 평균 영업이익률: 3.1%


In [11]:
# 5. 데이터 저장 및 종합 리포트
if 'stocks_clean' in locals() and not stocks_clean.empty:
    print("\n💾 5. 데이터 저장 및 종합 리포트")
    print("-" * 40)
    
    # 현재 시각
    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Excel 파일로 저장
    excel_filename = f"LG_Global_Competitors_Analysis_{current_time}.xlsx"
    excel_path = f"c:\\Users\\lgdx\\LG_DX_School\\02_DX_Methodology\\LG_Financial_Analysis\\{excel_filename}"
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # 전체 데이터
        stocks_clean.to_excel(writer, sheet_name='전체데이터', index=False)
        
        # 시가총액 순위
        if 'market_cap_data' in locals():
            market_cap_data[['회사명', '국가', 'market_cap_billion']].to_excel(
                writer, sheet_name='시가총액순위', index=False
            )
        
        # 밸류에이션 데이터
        valuation_summary = stocks_clean[['회사명', '국가', 'pe_ratio', 'pb_ratio']].dropna()
        if not valuation_summary.empty:
            valuation_summary.to_excel(writer, sheet_name='밸류에이션', index=False)
        
        # 수익성 데이터
        profitability_summary = stocks_clean[['회사명', '국가', 'roe', 'operating_margin', 'profit_margin']].dropna()
        if not profitability_summary.empty:
            profitability_summary.to_excel(writer, sheet_name='수익성', index=False)
        
        # 국가별 분석
        if 'country_analysis' in locals():
            country_analysis.to_excel(writer, sheet_name='국가별분석')
    
    print(f"✅ Excel 파일 저장 완료: {excel_filename}")
    
    # JSON 파일로도 저장
    json_filename = f"LG_Global_Competitors_Data_{current_time}.json"
    json_path = f"c:\\Users\\lgdx\\LG_DX_School\\02_DX_Methodology\\LG_Financial_Analysis\\{json_filename}"
    
    stocks_clean.to_json(json_path, orient='records', indent=2, force_ascii=False)
    print(f"✅ JSON 파일 저장 완료: {json_filename}")
    
    # 종합 리포트
    print(f"\n📋 종합 분석 리포트")
    print("=" * 50)
    
    total_companies = len(stocks_clean)
    successful_data = stocks_clean['market_cap_usd'].notna().sum()
    
    print(f"📊 데이터 수집 현황:")
    print(f"   • 분석 대상 기업: {total_companies}개")
    print(f"   • 시가총액 데이터: {successful_data}개")
    print(f"   • 데이터 수집 성공률: {(successful_data/total_companies*100):.1f}%")
    
    if 'LG전자' in stocks_clean['회사명'].values:
        lg_data = stocks_clean[stocks_clean['회사명'] == 'LG전자'].iloc[0]
        print(f"\n🇰🇷 LG전자 현재 상태:")
        print(f"   • 현재 주가: {lg_data['current_price']:.0f} {lg_data['currency']}")
        print(f"   • 시가총액: ${lg_data['market_cap_usd']/1e9:.1f}B")
        if pd.notna(lg_data['pe_ratio']):
            print(f"   • P/E Ratio: {lg_data['pe_ratio']:.1f}x")
        if pd.notna(lg_data['pb_ratio']):
            print(f"   • P/B Ratio: {lg_data['pb_ratio']:.1f}x")
        if pd.notna(lg_data['roe']):
            print(f"   • ROE: {lg_data['roe']:.1f}%")
    
    print(f"\n💡 주요 인사이트:")
    
    # 시가총액 1위 기업
    if 'market_cap_data' in locals() and not market_cap_data.empty:
        top_company = market_cap_data.iloc[0]
        print(f"   • 시가총액 1위: {top_company['회사명']} ({top_company['국가']}) - ${top_company['market_cap_billion']:.1f}B")
    
    # ROE 1위 기업
    if not stocks_clean['roe'].isna().all():
        top_roe_company = stocks_clean.loc[stocks_clean['roe'].idxmax()]
        print(f"   • ROE 1위: {top_roe_company['회사명']} ({top_roe_company['국가']}) - {top_roe_company['roe']:.1f}%")
    
    # 가장 저평가된 기업 (P/E 기준)
    pe_data = stocks_clean[stocks_clean['pe_ratio'].notna() & (stocks_clean['pe_ratio'] > 0)]
    if not pe_data.empty:
        undervalued_company = pe_data.loc[pe_data['pe_ratio'].idxmin()]
        print(f"   • 저평가 기업: {undervalued_company['회사명']} ({undervalued_company['국가']}) - P/E {undervalued_company['pe_ratio']:.1f}x")
    
    print(f"\n🎯 분석 완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
else:
    print("❌ 저장할 데이터가 없습니다.")


💾 5. 데이터 저장 및 종합 리포트
----------------------------------------
✅ Excel 파일 저장 완료: LG_Global_Competitors_Analysis_20250801_103037.xlsx
✅ JSON 파일 저장 완료: LG_Global_Competitors_Data_20250801_103037.json

📋 종합 분석 리포트
📊 데이터 수집 현황:
   • 분석 대상 기업: 10개
   • 시가총액 데이터: 10개
   • 데이터 수집 성공률: 100.0%

🇰🇷 LG전자 현재 상태:
   • 현재 주가: 75700 KRW
   • 시가총액: $9.7B

💡 주요 인사이트:
   • 시가총액 1위: General Electric (미국) - $287.5B
   • ROE 1위: General Electric (미국) - 39.6%
   • 저평가 기업: Panasonic (일본) - P/E 9.3x

🎯 분석 완료 시각: 2025-08-01 10:30:37
✅ Excel 파일 저장 완료: LG_Global_Competitors_Analysis_20250801_103037.xlsx
✅ JSON 파일 저장 완료: LG_Global_Competitors_Data_20250801_103037.json

📋 종합 분석 리포트
📊 데이터 수집 현황:
   • 분석 대상 기업: 10개
   • 시가총액 데이터: 10개
   • 데이터 수집 성공률: 100.0%

🇰🇷 LG전자 현재 상태:
   • 현재 주가: 75700 KRW
   • 시가총액: $9.7B

💡 주요 인사이트:
   • 시가총액 1위: General Electric (미국) - $287.5B
   • ROE 1위: General Electric (미국) - 39.6%
   • 저평가 기업: Panasonic (일본) - P/E 9.3x

🎯 분석 완료 시각: 2025-08-01 10:30:37


## 🎯 결론 및 향후 계획

### 📊 **분석 결과 요약**
이번 분석을 통해 LG전자와 글로벌 주요 경쟁사들의 실시간 재무 데이터를 성공적으로 수집하고 비교 분석했습니다.

### 💡 **주요 발견사항**
1. **시가총액 포지션**: LG전자의 글로벌 시장에서의 규모 확인
2. **밸류에이션 비교**: P/E, P/B 비율을 통한 상대적 저평가/고평가 분석
3. **수익성 벤치마크**: ROE, 영업이익률 등 핵심 수익성 지표 비교
4. **국가별 경쟁력**: 각 국가별 기업들의 평균적 경쟁력 수준 파악

### 🚀 **향후 개선 계획**
1. **데이터 확장**: 
   - 더 많은 경쟁사 추가 (인도, 브라질 등)
   - 분기별 실적 추이 분석
   - 기술적 분석 지표 추가

2. **API 활용도 증대**:
   - Alpha Vantage API 본격 활용
   - SEC 공시 데이터 연동
   - 실시간 뉴스 및 공시 모니터링

3. **자동화 구축**:
   - 일/주/월 단위 자동 업데이트
   - 이메일/대시보드 리포트 자동 발송
   - 임계값 기반 알림 시스템

### 📁 **생성된 파일들**
- **Excel 분석 파일**: 종합 데이터 및 차트
- **JSON 데이터 파일**: API 연동용 구조화된 데이터
- **시각화 차트**: 인터랙티브 Plotly 차트들

### 🔄 **정기 업데이트 권장사항**
- **일일**: 주가 및 시가총액 모니터링
- **주간**: 밸류에이션 지표 변화 추적
- **월간**: 재무 성과 지표 종합 분석
- **분기**: 경쟁사 포트폴리오 재구성

---
*📈 이 분석을 통해 LG전자의 글로벌 경쟁 포지션을 객관적으로 파악하고, 데이터 기반 전략 수립의 기초 자료로 활용하시기 바랍니다.*